# Automated AI Bakery Verification

In this notebook, the procedure developed in the exploratory verification pilot will be applied with those findings. The bakery candidates will be processed in descending ranks, with each verifcation result saved and considered for the overall classification. Businesses will be processed in API batches and the process will be resumed across multiple days due to API rate limits. Previous verification results are loaded automatically so that businesses with completed verification are not resubmitted. The results will later be included in a final bakery name classification results csv.

In [4]:
from pathlib import Path
from getpass import getpass
from datetime import datetime

import json
import re
import pandas as pd

from google import genai
from google.genai import types, errors

# Verification settings
MODEL = "gemini-2.5-flash"

BUSINESSES_PER_REQUEST = 10
MAX_REQUESTS_PER_RUN = 400

# Optional rank range
RANK_START = 1
RANK_END = None


# Paths
FHRS_DATA_DATE = "2026-07-23"

DATA_FOLDER = Path("../data/business/interim")
VERIFICATION_FOLDER = (DATA_FOLDER / "ai_verification")
VERIFICATION_FOLDER.mkdir(parents=True, exist_ok=True)

RANKED_FHRS_PATH = (DATA_FOLDER / f"london_fhrs_ranked_establishments_{FHRS_DATA_DATE}.csv")

RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results.csv")
BATCH_LOG_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_batches.jsonl")
ERROR_LOG_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_errors.jsonl")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")


In [ ]:
fhrs_ranked = pd.read_csv(RANKED_FHRS_PATH)

verification_queue = (fhrs_ranked[fhrs_ranked["BakeryRank"].notna()]
    .sort_values(["BakeryRank", "FHRSID"])
    .drop_duplicates(subset="BusinessNameClean")
    .rename(columns= {"FHRSID":"FHRSIDRep"})
    .reset_index(drop=True))

if RANK_START is not None:
    verification_queue = verification_queue[verification_queue["BakeryRank"] >= RANK_START]

if RANK_END is not None:
    verification_queue = verification_queue[verification_queue["BakeryRank"] <= RANK_END]

verification_queue = (verification_queue.sort_values("BakeryRank").reset_index(drop=True))

print(
    f"Verification queue: {len(verification_queue)} unique businesses\n"
    f"Rank range: {int(verification_queue['BakeryRank'].min())} – {int(verification_queue['BakeryRank'].max())}"
)

Verification queue: 65799 unique businesses
Rank range: 1 – 65799


In [8]:
if RESULTS_PATH.exists():
    previous_results = pd.read_csv(RESULTS_PATH)
    completed_names = set(previous_results["BusinessNameClean"].dropna())

else:
    previous_results = pd.DataFrame()
    completed_names = set()

remaining_queue = verification_queue[~verification_queue["BusinessNameClean"].isin(completed_names)].copy()
remaining_queue = (remaining_queue.sort_values("BakeryRank").reset_index(drop=True))

max_reqs_per_run = (BUSINESSES_PER_REQUEST * MAX_REQUESTS_PER_RUN)

run_queue = (remaining_queue.head(max_reqs_per_run).copy().reset_index(drop=True))

requests_planned = (len(run_queue) + BUSINESSES_PER_REQUEST- 1) // BUSINESSES_PER_REQUEST

print(
f"""Previously completed: {len(completed_names)}
Remaining businesses: {len(remaining_queue)}
Businesses selected this run: {len(run_queue)}
Maximum API requests this run: {requests_planned}""")

Previously completed: 0
Remaining businesses: 65799
Businesses selected this run: 4000
Maximum API requests this run: 400


In [9]:
api_key = getpass("Gemini API key: ")

client = genai.Client(api_key=api_key)

ValueError: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.

In [ ]:
def clean_value(value):
    if pd.isna(value):
        return "Unknown"
    return str(value)

def build_verification_prompt(batch):
    batch = (batch.reset_index(drop=True))

    business_blocks = []

    for i, row in batch.iterrows():
        business_blocks.append(
            f"""

BUSINESS {i + 1}
Name: {clean_value(row["BusinessName"])}
FHRS ID: {clean_value(row["FHRSIDRep"])}
FHRS type: {clean_value(row["BusinessType"])}
Postcode: {clean_value(row["PostCode"])}
Local authority: {clean_value(row["LocalAuthorityName"])}
""".strip())

    business_text = "\n\n".join(business_blocks)

    prompt = f"""
I am verifying London food businesses for a dissertation dataset.

Investigate EACH business independently using Google Search.

For each business, first try to identify the specific establishment using the
business name together with the supplied FHRS ID, postcode, local authority and
FHRS business type.

Evidence from a similarly named business in another location is not valid.

Classification rules:

BAKERY
Use this when the business primarily produces or specialises in selling baked
goods such as bread, pastries, cakes, biscuits, cookies, pies or similar bakery
products. This includes bakeries, patisseries and specialist cake businesses.

NOT_BAKERY
Use this when the business is primarily another type of establishment, even if
it also sells some baked products. Examples include ordinary cafes,
restaurants, supermarkets, convenience stores, caterers, confectionery shops,
ice cream shops and dessert parlours.

A dessert business should not be classified as a bakery simply because it sells
waffles, pancakes, crepes or other desserts. Bakery products must themselves be
a main part of the business.

UNCLEAR
Use this when:
- the supplied establishment cannot be confidently identified;
- evidence does not establish its main activity;
- search results appear to refer to another business;
- or reliable sources conflict.

Important:
- Search for every numbered business separately.
- Do not classify from the business name alone.
- Do not use similarly named businesses as evidence.
- Do not infer products that are not supported by evidence.
- It is better to return UNCLEAR than to guess.

{business_text}

Return exactly one line for every business, in the same order.

Use this exact format with | between each field:

1 | YES | BAKERY | short evidence-based reason
2 | UNCLEAR | UNCLEAR | short evidence-based reason

The four fields are:
business number | location match | verdict | reason

LOCATION MATCH must be YES or UNCLEAR.
VERDICT must be BAKERY, NOT_BAKERY or UNCLEAR.

Do not add an introduction, conclusion, headings or markdown formatting.
""".strip()

    return prompt

In [12]:
def verify_current_batch(batch):

    prompt = build_verification_prompt(batch)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())]
        ))

    return prompt, response

In [ ]:
def get_grounding_info(response):

    if not response.candidates:
        return [], []

    grounding = response.candidates[0].grounding_metadata

    if grounding is None:
        return [], []

    search_queries = list(grounding.web_search_queries or [])

    source_urls = []

    for chunk in grounding.grounding_chunks or []:
        if chunk.web:
            source_urls.append(chunk.web.uri)

    # Remove duplicate URLs
    source_urls = list(dict.fromkeys(source_urls))
    return search_queries, source_urls